# World Happiness Report 2026 — Comprehensive Global Analysis
### What Makes Nations Happy? A Data-Driven Exploration of 147 Countries

---

**Author:** Hassan Ali  
**Role:** Data Scientist & Machine Learning Engineer  
**LinkedIn:** [linkedin.com/in/hassan-ali-datascientist](https://linkedin.com/in/hassan-ali-datascientist)  
**GitHub:** [github.com/hassan-ali786](https://github.com/hassan-ali786)  
**Portfolio:** [hassanali-portfolio.vercel.app](https://hassanali-portfolio.vercel.app/)

---

## About This Notebook

The World Happiness Report 2026 — published on March 20, 2026 by the Wellbeing Research Centre at the University of Oxford — ranks 147 countries by how happy their citizens perceive themselves to be. The scores are based on the Cantril Ladder question from the Gallup World Poll, averaged over 2023–2025.

This notebook performs a comprehensive analysis of the 2026 data to answer:

- Which countries are the happiest — and which are the least happy in 2026?
- Which of the six factors — GDP, social support, health, freedom, generosity, corruption — drives happiness the most?
- Does money buy happiness? What does the GDP vs happiness relationship look like?
- How do world regions differ in happiness levels and drivers?
- What can a machine learning model tell us about the formula for national happiness?
- What are the biggest surprises and key findings of the 2026 report?

---

## Table of Contents

1. Environment Setup
2. Data Loading and Overview
3. Global Happiness Distribution
4. Top and Bottom Countries — 2026 Rankings
5. Regional Analysis
6. Factor Correlation Analysis
7. GDP vs Happiness — Does Money Buy Happiness?
8. Multi-Factor Scatter Analysis
9. Predictive Modeling — What Drives Happiness?
10. Key Insights and Conclusions

---
*If this analysis adds value to your work, an upvote is appreciated — it helps more people find this resource.*

---
## Section 1: Environment Setup

In [ ]:
# Core
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Visualization
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

# Machine Learning
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import r2_score, mean_squared_error

# Plot Configuration
BLUE    = '#2C7BB6'
RED     = '#D73027'
GREEN   = '#1A9641'
ORANGE  = '#FDAE61'
PALETTE = [BLUE, RED, GREEN, ORANGE, '#ABD9E9', '#F46D43', '#74ADD1', '#A50026', '#313695', '#FFFFBF']
REGION_COLORS = {
    'Western Europe'                      : '#2C7BB6',
    'North America and ANZ'               : '#1A9641',
    'Latin America and Caribbean'         : '#FDAE61',
    'Central and Eastern Europe'          : '#74ADD1',
    'East Asia'                           : '#F46D43',
    'Southeast Asia'                      : '#D73027',
    'South Asia'                          : '#A50026',
    'Middle East and North Africa'        : '#984EA3',
    'Sub-Saharan Africa'                  : '#FF7F00',
    'Commonwealth of Independent States'  : '#999999'
}

plt.rcParams.update({
    'figure.dpi'        : 120,
    'axes.spines.top'   : False,
    'axes.spines.right' : False,
    'axes.titlesize'    : 13,
    'axes.titleweight'  : 'bold',
    'axes.labelsize'    : 11,
    'font.family'       : 'serif'
})
sns.set_style('whitegrid')

print('Environment configured successfully.')
print(f'NumPy  : {np.__version__}')
print(f'Pandas : {pd.__version__}')

---
## Section 2: Data Loading and Overview

The dataset contains 147 countries with 10 columns — rank, country, region, happiness score, and six contributing factors. Each row represents one country's data averaged over 2023–2025.

In [ ]:
# Load Dataset
df = pd.read_csv('/kaggle/input/world-happiness-report-2026/world_happiness_2026.csv')

print('Dataset loaded successfully.')
print(f'  Rows    : {df.shape[0]}')
print(f'  Columns : {df.shape[1]}')
print(f'  Regions : {df["region"].nunique()}')
print()
df.head(10)

In [ ]:
# Column Descriptions
col_desc = {
    'rank'                    : 'Happiness ranking (1 = happiest)',
    'country'                 : 'Country name',
    'region'                  : 'World region',
    'score'                   : 'Happiness score — Cantril Ladder (0–10)',
    'gdp_per_capita'          : 'Economic output per person (log scale)',
    'social_support'          : 'Perceived availability of social support',
    'healthy_life_expectancy' : 'Expected years of healthy life',
    'freedom'                 : 'Freedom to make life choices',
    'generosity'              : 'Charitable giving behavior',
    'corruption'              : 'Perception of low corruption (higher = less corrupt)'
}
print('Column Descriptions:')
for col, desc in col_desc.items():
    print(f'  {col:<28} : {desc}')

print(f'\nBasic Statistics:')
df.describe().round(3)

In [ ]:
# Missing Values and Data Quality
print('Missing Values:')
print(df.isnull().sum())
print()
print(f'Score Range : {df["score"].min():.3f} — {df["score"].max():.3f}')
print(f'Global Mean : {df["score"].mean():.3f}')
print(f'Global Median : {df["score"].median():.3f}')
print()
print('Countries per Region:')
print(df['region'].value_counts().to_string())

---
## Section 3: Global Happiness Distribution

Before examining individual countries, we first understand how happiness is distributed globally. The Cantril Ladder runs from 0 to 10, but in practice, no country falls below 1.4 or above 7.8 — the real-world range reflects the limits of human social organization.

In [ ]:
# Add happiness tier
def assign_tier(score):
    if score >= 7.0:   return 'Very Happy (7+)'
    elif score >= 6.0: return 'Happy (6–7)'
    elif score >= 5.0: return 'Moderate (5–6)'
    elif score >= 4.0: return 'Unhappy (4–5)'
    else:              return 'Very Unhappy (<4)'

df['tier'] = df['score'].apply(assign_tier)

tier_colors = {
    'Very Happy (7+)'   : '#1A9641',
    'Happy (6–7)'       : '#A6D96A',
    'Moderate (5–6)'    : '#FDAE61',
    'Unhappy (4–5)'     : '#F46D43',
    'Very Unhappy (<4)' : '#D73027'
}

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Histogram
axes[0].hist(df['score'], bins=25, color=BLUE, edgecolor='white', alpha=0.85)
axes[0].axvline(df['score'].mean(),   color=RED,   linestyle='--', linewidth=2,
                label=f'Mean: {df["score"].mean():.2f}')
axes[0].axvline(df['score'].median(), color=GREEN, linestyle='-.', linewidth=2,
                label=f'Median: {df["score"].median():.2f}')
axes[0].set_title('Global Happiness Score Distribution — 2026')
axes[0].set_xlabel('Happiness Score (Cantril Ladder 0–10)')
axes[0].set_ylabel('Number of Countries')
axes[0].legend(fontsize=10)

# Tier distribution
tier_order  = ['Very Happy (7+)', 'Happy (6–7)', 'Moderate (5–6)', 'Unhappy (4–5)', 'Very Unhappy (<4)']
tier_counts = df['tier'].value_counts().reindex(tier_order)
colors_pie  = [tier_colors[t] for t in tier_order]
axes[1].pie(tier_counts.values, labels=tier_order, colors=colors_pie,
            autopct='%1.1f%%', startangle=90, pctdistance=0.82)
axes[1].set_title('Countries by Happiness Tier — 2026')

fig.suptitle('2026 World Happiness — Global Overview', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('Interpretation:')
print(f'  Global mean score  : {df["score"].mean():.3f}')
print(f'  Standard deviation : {df["score"].std():.3f}')
print(f'  Score gap (top–bottom): {df["score"].max() - df["score"].min():.3f} points')
print()
print('The 2026 data shows substantial inequality in global happiness.')
print('The gap between the happiest country (Finland: 7.764) and the least happy')
print('(Afghanistan: 1.446) is 6.318 points — nearly two thirds of the full scale.')

---
## Section 4: Top and Bottom Countries — 2026 Rankings

The 2026 rankings carry a significant surprise: Costa Rica breaks into the Top 5 — the highest rank ever achieved by a Latin American country. Meanwhile, Nordic nations continue their long dominance at the top.

In [ ]:
top20    = df.head(20)
bottom15 = df.tail(15).sort_values('score', ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(17, 8))

# Top 20
colors_top = [REGION_COLORS.get(r, BLUE) for r in top20['region']]
bars1 = axes[0].barh(top20['country'][::-1], top20['score'][::-1],
                     color=colors_top[::-1], edgecolor='white')
axes[0].set_title('Top 20 Happiest Countries — 2026')
axes[0].set_xlabel('Happiness Score')
axes[0].set_xlim(0, 9)
for i, (country, score) in enumerate(zip(top20['country'][::-1], top20['score'][::-1])):
    axes[0].text(score + 0.05, i, f'{score:.3f}', va='center', fontsize=8.5)

# Bottom 15
colors_bot = [REGION_COLORS.get(r, RED) for r in bottom15['region']]
bars2 = axes[1].barh(bottom15['country'], bottom15['score'],
                     color=colors_bot, edgecolor='white')
axes[1].set_title('Bottom 15 Least Happy Countries — 2026')
axes[1].set_xlabel('Happiness Score')
axes[1].set_xlim(0, 5)
for i, (country, score) in enumerate(zip(bottom15['country'], bottom15['score'])):
    axes[1].text(score + 0.02, i, f'{score:.3f}', va='center', fontsize=8.5)

fig.suptitle('World Happiness Report 2026 — Ranking Extremes', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('2026 Key Ranking Highlights:')
print('  - Finland retains #1 for the ninth consecutive year (score: 7.764)')
print('  - Costa Rica at #4 — highest ever rank for a Latin American country')
print('  - USA at #23 — below UAE (#21) and Saudi Arabia (#22) for the first time')
print('  - UK at #25, Canada at #15')
print('  - Afghanistan remains last at 1.446 — score less than 1/5 of Finland')
print('  - All bottom 15 countries are in Sub-Saharan Africa or conflict zones')

---
## Section 5: Regional Analysis

World regions share historical, cultural, and institutional characteristics that shape collective wellbeing. Regional analysis reveals structural patterns that transcend individual country stories.

In [ ]:
region_stats = df.groupby('region')['score'].agg(['mean', 'median', 'std', 'count'])
region_stats = region_stats.sort_values('mean', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(17, 6))

# Regional mean bar chart
colors_reg = [REGION_COLORS.get(r, BLUE) for r in region_stats.index]
axes[0].barh(region_stats.index[::-1], region_stats['mean'][::-1],
             color=colors_reg[::-1], edgecolor='white')
axes[0].set_title('Mean Happiness Score by Region — 2026')
axes[0].set_xlabel('Mean Happiness Score')
axes[0].set_xlim(0, 8)
for i, (reg, val) in enumerate(zip(region_stats.index[::-1], region_stats['mean'][::-1])):
    axes[0].text(val + 0.05, i, f'{val:.3f}', va='center', fontsize=9)

# Box plot
region_order = region_stats.index.tolist()
bp_data = [df[df['region'] == r]['score'].values for r in region_order]
bp = axes[1].boxplot(bp_data, vert=False, patch_artist=True,
                     medianprops=dict(color='black', linewidth=2))
for patch, region in zip(bp['boxes'], region_order):
    patch.set_facecolor(REGION_COLORS.get(region, BLUE))
    patch.set_alpha(0.75)
axes[1].set_yticks(range(1, len(region_order) + 1))
axes[1].set_yticklabels(region_order, fontsize=8)
axes[1].set_title('Score Distribution by Region — 2026')
axes[1].set_xlabel('Happiness Score')

fig.suptitle('Regional Happiness Analysis — World Happiness Report 2026', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('Regional Summary:')
print(region_stats.round(3).to_string())
print()
print('Key Observation:')
print('Western Europe leads with a mean of ~7.0 — driven by Nordic nations.')
print('Sub-Saharan Africa scores lowest — impacted by poverty, conflict, and weak institutions.')
print('Latin America outperforms its GDP ranking — reflecting strong social bonds and community.')

---
## Section 6: Factor Correlation Analysis

The six factors explain why some countries score higher than others relative to a hypothetical baseline called Dystopia — a country with the world's worst values on every factor. We examine which factors correlate most strongly with overall happiness.

In [ ]:
feature_cols = ['gdp_per_capita', 'social_support', 'healthy_life_expectancy',
                'freedom', 'generosity', 'corruption']

factor_labels = {
    'gdp_per_capita'          : 'GDP per Capita',
    'social_support'          : 'Social Support',
    'healthy_life_expectancy' : 'Healthy Life Expectancy',
    'freedom'                 : 'Freedom',
    'generosity'              : 'Generosity',
    'corruption'              : 'Corruption (low = more corrupt)'
}

correlations = df[feature_cols].corrwith(df['score']).sort_values()
correlations.index = [factor_labels[c] for c in correlations.index]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Correlation bar
colors_corr = [GREEN if v > 0 else RED for v in correlations.values]
axes[0].barh(correlations.index, correlations.values, color=colors_corr, edgecolor='white')
axes[0].axvline(0, color='black', linewidth=0.8)
axes[0].set_title('Factor Correlation with Happiness Score')
axes[0].set_xlabel('Pearson Correlation Coefficient')
for i, (feat, val) in enumerate(zip(correlations.index, correlations.values)):
    offset = 0.01 if val >= 0 else -0.01
    ha = 'left' if val >= 0 else 'right'
    axes[0].text(val + offset, i, f'{val:.3f}', va='center', ha=ha, fontsize=9)

# Full heatmap
corr_matrix = df[feature_cols + ['score']].corr()
corr_matrix.index   = [factor_labels.get(c, c) for c in corr_matrix.index]
corr_matrix.columns = [factor_labels.get(c, c) for c in corr_matrix.columns]
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='RdYlGn', center=0, vmin=-1, vmax=1,
            linewidths=0.5, annot_kws={'size': 8}, ax=axes[1])
axes[1].set_title('Full Correlation Matrix — All Factors')

fig.suptitle('Happiness Factor Correlation Analysis — 2026', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

strongest = correlations.abs().idxmax()
print(f'Strongest factor: {strongest} (r = {correlations[strongest]:.3f})')
print()
print('Factor Rankings by Correlation Strength:')
for feat, val in correlations.abs().sort_values(ascending=False).items():
    direction = 'positive' if correlations[feat] > 0 else 'negative'
    print(f'  {feat:<35} r = {correlations[feat]:>7.3f}  ({direction})')

---
## Section 7: GDP vs Happiness — Does Money Buy Happiness?

This question has been debated in economics and psychology for decades. The 2026 data offers a clear answer: yes, economic prosperity strongly predicts happiness — but with diminishing returns at higher income levels, and notable exceptions where countries outperform their GDP ranking.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Scatter — GDP vs Score by Region
for region, color in REGION_COLORS.items():
    mask = df['region'] == region
    if mask.sum() > 0:
        axes[0].scatter(df[mask]['gdp_per_capita'], df[mask]['score'],
                        label=region, color=color, alpha=0.75, s=45, edgecolors='none')

# Trend line
z = np.polyfit(df['gdp_per_capita'], df['score'], 1)
p = np.poly1d(z)
x_line = np.linspace(df['gdp_per_capita'].min(), df['gdp_per_capita'].max(), 100)
axes[0].plot(x_line, p(x_line), color='black', linewidth=1.8, linestyle='--', label='Trend line')

# Annotate key countries
highlights = ['Finland', 'Afghanistan', 'Costa Rica', 'United States', 'India', 'Japan']
for _, row in df[df['country'].isin(highlights)].iterrows():
    axes[0].annotate(row['country'], (row['gdp_per_capita'], row['score']),
                     textcoords='offset points', xytext=(5, 3), fontsize=7.5)

corr_gdp = df['gdp_per_capita'].corr(df['score'])
axes[0].set_title('GDP per Capita vs Happiness Score — 2026')
axes[0].set_xlabel('GDP per Capita (log scale)')
axes[0].set_ylabel('Happiness Score')
axes[0].annotate(f'r = {corr_gdp:.3f}', xy=(0.05, 0.92), xycoords='axes fraction',
                 fontsize=11, bbox=dict(boxstyle='round', facecolor='white', edgecolor='gray'))
axes[0].legend(fontsize=6.5, ncol=2, loc='lower right')

# GDP Quartile Analysis
df['gdp_quartile'] = pd.qcut(df['gdp_per_capita'], q=4,
                              labels=['Q1 Poorest', 'Q2', 'Q3', 'Q4 Richest'])
gdp_q = df.groupby('gdp_quartile')['score'].mean()
bars = axes[1].bar(gdp_q.index, gdp_q.values,
                   color=[RED, ORANGE, '#A6D96A', GREEN], edgecolor='white')
axes[1].set_title('Mean Happiness Score by GDP Quartile — 2026')
axes[1].set_xlabel('GDP per Capita Quartile')
axes[1].set_ylabel('Mean Happiness Score')
axes[1].set_ylim(0, 8)
for bar, val in zip(bars, gdp_q.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                 f'{val:.3f}', ha='center', fontweight='bold')

fig.suptitle('Economic Prosperity and Happiness — 2026', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

q1 = df[df['gdp_quartile'] == 'Q1 Poorest']['score'].mean()
q4 = df[df['gdp_quartile'] == 'Q4 Richest']['score'].mean()
print(f'Mean score — Poorest quartile  : {q1:.3f}')
print(f'Mean score — Richest quartile  : {q4:.3f}')
print(f'Happiness gap Q1 to Q4         : {q4 - q1:.3f} points')
print()
print('Notable 2026 Finding:')
print('Costa Rica (rank 4) outperforms its GDP ranking significantly.')
print('Mexico (rank 12) scores higher than many richer European nations.')
print('This confirms that community, social trust, and freedom matter')
print('as much as economic output at comparable income levels.')

---
## Section 8: Multi-Factor Scatter Analysis

We examine all six happiness factors simultaneously — each plotted against the happiness score with a trend line — to understand the direction and strength of each relationship.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 10))
axes = axes.flatten()

for i, col in enumerate(feature_cols):
    corr_val = df[col].corr(df['score'])
    color    = GREEN if corr_val > 0 else RED

    axes[i].scatter(df[col], df['score'], alpha=0.5, color=color, s=25, edgecolors='none')

    z = np.polyfit(df[col], df['score'], 1)
    p = np.poly1d(z)
    x_l = np.linspace(df[col].min(), df[col].max(), 100)
    axes[i].plot(x_l, p(x_l), color='black', linewidth=1.5, linestyle='--')

    axes[i].set_title(f'{factor_labels[col]}\n(r = {corr_val:.3f})')
    axes[i].set_xlabel(factor_labels[col])
    if i % 3 == 0:
        axes[i].set_ylabel('Happiness Score')

    axes[i].annotate(f'r = {corr_val:.3f}',
                     xy=(0.05, 0.90), xycoords='axes fraction', fontsize=10,
                     bbox=dict(boxstyle='round', facecolor='white', edgecolor='gray', alpha=0.8))

fig.suptitle('All Six Happiness Factors vs Score — 2026', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('Factor Interpretations:')
print('  GDP per Capita          — Strongest positive predictor. Higher income = higher happiness.')
print('  Social Support          — Nearly as strong as GDP. Community matters enormously.')
print('  Healthy Life Expectancy — Strong positive link. Health enables life satisfaction.')
print('  Freedom                 — Moderate positive. Autonomy improves wellbeing.')
print('  Generosity              — Weakest factor. Giving behavior has limited national impact.')
print('  Corruption              — Negative at lower values — higher corruption = lower happiness.')

---
## Section 9: Predictive Modeling — What Drives Happiness?

We train machine learning models to quantify how well the six reported factors predict happiness scores. Feature importance from the best model reveals which factors the algorithm considers most influential — providing a data-driven perspective beyond correlation.

In [ ]:
X = df[feature_cols]
y = df['score']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

models = {
    'Linear Regression'  : LinearRegression(),
    'Ridge Regression'   : Ridge(alpha=1.0),
    'Random Forest'      : RandomForestRegressor(n_estimators=200, random_state=42),
    'Gradient Boosting'  : GradientBoostingRegressor(n_estimators=200, learning_rate=0.05, random_state=42)
}

results = {}
print(f'{"Model":<25} {"CV R²":>10} {"Test R²":>10} {"Test RMSE":>12}')
print('-' * 60)

for name, model in models.items():
    cv_r2     = cross_val_score(model, X_train, y_train, cv=5, scoring='r2').mean()
    model.fit(X_train, y_train)
    y_pred    = model.predict(X_test)
    test_r2   = r2_score(y_test, y_pred)
    test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    results[name] = {'CV R2': cv_r2, 'Test R2': test_r2, 'RMSE': test_rmse}
    print(f'{name:<25} {cv_r2:>10.4f} {test_r2:>10.4f} {test_rmse:>12.4f}')

In [ ]:
# Feature Importance + Model Comparison
gb_model  = models['Gradient Boosting']
feat_imp  = pd.Series(gb_model.feature_importances_, index=feature_cols)
feat_imp.index = [factor_labels[c] for c in feat_imp.index]
feat_imp  = feat_imp.sort_values(ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Feature importance
colors_imp = [GREEN if v > feat_imp.median() else '#A6D96A' for v in feat_imp.values]
axes[0].barh(feat_imp.index, feat_imp.values, color=colors_imp, edgecolor='white')
axes[0].set_title('Feature Importance — Gradient Boosting Model')
axes[0].set_xlabel('Importance Score')
for i, val in enumerate(feat_imp.values):
    axes[0].text(val + 0.002, i, f'{val:.3f}', va='center', fontsize=9)

# Model comparison
model_names = list(results.keys())
test_r2s    = [results[m]['Test R2'] for m in model_names]
axes[1].bar(model_names, test_r2s,
            color=[BLUE, '#74ADD1', GREEN, RED], edgecolor='white')
axes[1].set_title('Model Comparison — Test R² Score')
axes[1].set_ylabel('R² Score')
axes[1].set_ylim(0, 1)
axes[1].set_xticklabels(model_names, rotation=15, ha='right')
for i, val in enumerate(test_r2s):
    axes[1].text(i, val + 0.01, f'{val:.4f}', ha='center', fontweight='bold')

fig.suptitle('Predictive Modeling Results — World Happiness 2026', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

best_r2 = max(results.values(), key=lambda x: x['Test R2'])['Test R2']
print(f'Best model R² : {best_r2:.4f}')
print(f'The six factors explain {best_r2*100:.1f}% of happiness score variation.')
print('The remaining variance reflects cultural, historical, and psychological')
print('factors not captured in the six reported dimensions.')

---
## Section 10: Key Insights and Conclusions

### 2026 Major Findings

| Finding | Detail |
|---------|--------|
| **Finland #1 again** | Ninth consecutive year at the top — score: 7.764 |
| **Costa Rica breaks Top 5** | Rank 4 — highest ever for a Latin American country |
| **USA below Gulf states** | US at #23, below UAE (#21) and Saudi Arabia (#22) for the first time |
| **Youth happiness crisis** | In English-speaking countries, under-25 happiness fell 0.86 points over 20 years |
| **Social media link** | Students using social media 7+ hours/day show substantially lower wellbeing |
| **Nordic dominance** | All Top 6 are Nordic or Costa Rica — combining GDP, freedom, social support, and low corruption |
| **Afghanistan last** | Score of 1.446 — less than one fifth of Finland's score |

---

### What the Data Reveals About the Formula for National Happiness

**1. Economic prosperity matters — but it is not sufficient**  
GDP per Capita is the strongest single predictor (r ≈ 0.78). However, Costa Rica and Mexico outperform their GDP rankings substantially, showing that social cohesion and community can compensate for lower income.

**2. Social support is nearly as important as income**  
The second strongest factor. Countries where citizens feel they have someone to rely on in times of trouble consistently score higher — regardless of wealth level.

**3. Governance quality has a measurable impact**  
Low corruption and high freedom together explain a significant portion of the variance beyond GDP. Trust in institutions is not a luxury — it is a driver of wellbeing.

**4. Generosity is the weakest factor**  
Charitable giving behavior shows the weakest correlation with national happiness scores. This may reflect measurement challenges as much as a true absence of effect.

**5. The 2026 special focus — social media and youth wellbeing**  
The report's most striking finding is not about country rankings but about age. In five English-speaking countries (US, Canada, UK, Australia, New Zealand), young adults under 25 now score below the over-60 population in life satisfaction — a reversal of the historical age gradient that the report's editors link to heavy social media use.

---

### Limitations

- Scores are self-reported and influenced by cultural norms around expressing satisfaction
- The six factors are partially collinear — their individual causal effects are difficult to isolate
- National averages mask within-country inequality — a high average may conceal deep disparities
- Correlation does not imply causation — the report's authors are explicit on this point

---

### Data Source

World Happiness Report 2026. Helliwell, J. F., Layard, R., Sachs, J. D., De Neve, J.-E., Aknin, L. B., & Wang, S. (Eds.). University of Oxford: Wellbeing Research Centre, published March 20, 2026. Underlying data from the Gallup World Poll (2023–2025 three-year average).

---

If this analysis was valuable, please upvote — it helps more data scientists and researchers find this resource.

**Connect:**
- LinkedIn: [linkedin.com/in/hassan-ali-datascientist](https://linkedin.com/in/hassan-ali-datascientist)
- GitHub: [github.com/hassan-ali786](https://github.com/hassan-ali786)
- Portfolio: [hassanali-portfolio.vercel.app](https://hassanali-portfolio.vercel.app/)